# OpenShell Sandbox — agent demo

A minimal end-to-end walkthrough: a `ChatNVIDIA`-driven Deep Agent calls a
custom tool that runs inside an OpenShell sandbox. We use the **OpenShell
CLI** for sandbox lifecycle and policy authoring; the Python wrapper shows
up only as the agent's backend.

The story below is **default-deny → tool denied → swap policy → tool
succeeds** — the same flow the OpenShell project demos in its
[`sandbox-policy-quickstart`](https://github.com/NVIDIA/OpenShell/tree/main/examples/sandbox-policy-quickstart).


## Install the OpenShell CLI

OpenShell ships its CLI / gateway separately. See the
[OpenShell quickstart](https://docs.nvidia.com/openshell/latest/get-started/quickstart/)
for full instructions; on macOS / Linux:

```bash
uv tool install -U openshell
# or
curl -LsSf https://raw.githubusercontent.com/NVIDIA/OpenShell/main/install.sh | sh
```

The first `openshell sandbox create` call also boots a local gateway on
`127.0.0.1:8080`.


## Install the Python packages

In [ ]:
%pip install --upgrade --quiet \
    langchain-nvidia-ai-endpoints \
    langchain-nvidia-openshell \
    deepagents \
    openshell


## Access the NVIDIA API Catalog

To get access to the NVIDIA API Catalog, do the following:

1. Create a free account on the [NVIDIA API Catalog](https://build.nvidia.com/) and log in.
2. Click your profile icon, and then click **API Keys**. The **API Keys** page appears.
3. Click **Generate API Key**. The **Generate API Key** window appears.
4. Click **Generate Key**. You should see **API Key Granted**, and your key appears.
5. Copy and save the key as `NVIDIA_API_KEY`.
6. To verify your key, use the following code.


In [ ]:
import getpass
import os

if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    print("Valid NVIDIA_API_KEY already in environment. Delete to reset")
else:
    nvapi_key = getpass.getpass("NVAPI Key (starts with nvapi-): ")
    assert nvapi_key.startswith(
        "nvapi-"
    ), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key


## 1. Define the tool

A tiny tool that calls `curl https://api.github.com/zen` *inside the
sandbox* via `OpenShellSandbox.execute`. The agent decides when to use it.


In [ ]:
from langchain_core.tools import tool


def make_zen_tool(backend):
    @tool
    def github_zen() -> str:
        """Fetch a Zen of GitHub quote (a short proverb) from api.github.com.

        Use whenever the user asks for a GitHub Zen quote or a programming
        proverb. Returns the quote on success or an error message on failure.
        """
        result = backend.execute("curl -sSf --max-time 5 https://api.github.com/zen")
        if result.exit_code != 0:
            return f"Tool failed (exit {result.exit_code}): {result.output[:200]}"
        return result.output.strip()

    return github_zen


## 2. Default-deny: create the sandbox without network access

The policy below allows reads on `/usr`, `/lib`, `/etc`, etc., and a
restricted `sandbox` user — but **no `network_policies` block**, so the
egress proxy denies every outbound request by default.


In [ ]:
%%writefile policy-deny.yaml
version: 1

filesystem_policy:
  include_workdir: true
  read_only: [/usr, /lib, /etc, /app, /var/log, /proc/self, /dev/urandom]
  read_write: [/sandbox, /tmp, /dev/null]

landlock:
  compatibility: best_effort

process:
  run_as_user: sandbox
  run_as_group: sandbox


In [ ]:
!openshell sandbox create \
    --name openshell-demo \
    --policy policy-deny.yaml \
    --keep --no-tty -- bash
!openshell sandbox list


## 3. Run the agent — the tool should be denied

We attach to the running sandbox, build a `ChatNVIDIA` Deep Agent with our
single `github_zen` tool, and ask it to fetch a quote. The proxy will block
the curl; the tool returns that failure to the LLM, which reports it back.


In [ ]:
import openshell
from deepagents import create_deep_agent
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_openshell import OpenShellSandbox

# Host-side: open a client to the gateway, attach to the running sandbox.
client = openshell.SandboxClient.from_active_cluster()
backend = OpenShellSandbox(sandbox=client.get_session("openshell-demo"))

# Host-side: build the agent. Only `backend.execute(...)` inside the tool
# crosses into the sandbox.
agent = create_deep_agent(
    model=ChatNVIDIA(model="meta/llama-3.3-70b-instruct"),
    tools=[make_zen_tool(backend)],
    system_prompt=(
        "You are a careful assistant. When the user asks for a Zen "
        "quote, call the github_zen tool. If the tool reports an "
        "error, explain what went wrong in one sentence."
    ),
    backend=backend,
)

response = agent.invoke({"messages": [{
    "role": "user",
    "content": "Give me a GitHub Zen quote.",
}]})

print(response["messages"][-1].content)
client.close()


## 4. Tear down, swap the policy, recreate

The Filesystem and Process layers are locked at create time, so we delete
the sandbox and recreate it with a policy that explicitly allows
`api.github.com`.


In [ ]:
!openshell sandbox delete openshell-demo --wait

In [ ]:
%%writefile policy-allow.yaml
version: 1

filesystem_policy:
  include_workdir: true
  read_only: [/usr, /lib, /etc, /app, /var/log, /proc/self, /dev/urandom]
  read_write: [/sandbox, /tmp, /dev/null]

landlock:
  compatibility: best_effort

process:
  run_as_user: sandbox
  run_as_group: sandbox

network_policies:
  github_api:
    name: github-api-readonly
    endpoints:
      - host: api.github.com
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
    binaries:
      - { path: /usr/bin/curl }


In [ ]:
!openshell sandbox create \
    --name openshell-demo \
    --policy policy-allow.yaml \
    --keep --no-tty -- bash


## 5. Re-run the agent — same task, now succeeds

Identical agent, identical prompt. The only thing that changed is the
sandbox's policy.


In [ ]:
client = openshell.SandboxClient.from_active_cluster()
backend = OpenShellSandbox(sandbox=client.get_session("openshell-demo"))

agent = create_deep_agent(
    model=ChatNVIDIA(model="meta/llama-3.3-70b-instruct"),
    tools=[make_zen_tool(backend)],
    system_prompt=(
        "You are a careful assistant. When the user asks for a Zen "
        "quote, call the github_zen tool and pass through the result."
    ),
    backend=backend,
)

response = agent.invoke({"messages": [{
    "role": "user",
    "content": "Give me a GitHub Zen quote.",
}]})

print(response["messages"][-1].content)
client.close()


## 6. Cleanup

Delete the sandbox, remove the policy files, and confirm a clean slate.


In [ ]:
%%bash
openshell sandbox delete openshell-demo --wait || true
rm -f policy-deny.yaml policy-allow.yaml
echo "--- sandboxes ---"
openshell sandbox list
echo "--- policy files left behind ---"
ls policy-*.yaml 2>/dev/null || echo "(none)"


## Sources

- [OpenShell — sandbox-policy-quickstart](https://github.com/NVIDIA/OpenShell/tree/main/examples/sandbox-policy-quickstart)
  — the deny-then-allow demo this notebook mirrors.
- [OpenShell policy schema reference](https://docs.nvidia.com/openshell/latest/reference/policy-schema)
  — every YAML field in the policy files above.
- [LangChain Deep Agents — sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)
  — the sandbox-as-tool pattern.
- [`langchain-nvidia-openshell` README](../README.md) — the full API
  surface and the four-layer policy summary.
